# 1. LIBRARIES & ENVIRONMENT & DEVICE CONFIGURATION

In [18]:
import json
import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt
import sys
import ultralytics
from pathlib import Path
from ultralytics import YOLO


In [ ]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

if DEVICE == 0:
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: 0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


# 2.LOAD YAML & CLASS MAPPING

In [20]:
PROJECT_ROOT = Path.cwd().parent
DATA_YAML = PROJECT_ROOT / "datasets" / "data.yaml"

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

data_config

{'train': '../train/images',
 'val': '../valid/images',
 'test': '../test/images',
 'nc': 14,
 'names': ['beef',
  'chicken',
  'egg',
  'fritters',
  'fruit',
  'noodles',
  'other_carbs',
  'pork',
  'rice',
  'sambal',
  'seafood',
  'tempeh',
  'tofu',
  'vegetable'],
 'roboflow': {'workspace': '15_xii-mipa-2_martinus-alvin',
  'project': 'my-first-project-derb2',
  'version': 2,
  'license': 'CC BY 4.0',
  'url': 'https://universe.roboflow.com/15_xii-mipa-2_martinus-alvin/my-first-project-derb2/dataset/2'}}

In [21]:
names = data_config["names"]

print("Number of classes:", len(names))
print("\nClass mapping:")

for class_id, class_name in enumerate(names):
    print(f"{class_id}: {class_name}")

Number of classes: 14

Class mapping:
0: beef
1: chicken
2: egg
3: fritters
4: fruit
5: noodles
6: other_carbs
7: pork
8: rice
9: sambal
10: seafood
11: tempeh
12: tofu
13: vegetable


# 3. LOAD PRETRAINED YOLO

In [22]:
MODEL_NAME = "yolo11n-seg.pt"

model = YOLO(MODEL_NAME)

In [23]:
model.info()

YOLO11n-seg summary: 203 layers, 2,876,848 parameters, 0 gradients, 10.0 GFLOPs


(203, 2876848, 0, 9.9593344)

# 4. SANITY CHECK - 1 IMAGE

In [24]:
VAL_DIR = PROJECT_ROOT / "datasets" / "valid" / "images"

print("Validation directory:")
print(VAL_DIR.resolve())

print("Exists:", VAL_DIR.exists())

Validation directory:
D:\PREP_INTERN\nutrivision_pro\datasets\valid\images
Exists: True


In [25]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

val_images = sorted(
    [
        p for p in VAL_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]
)

print("Number of validation images:", len(val_images))

print("\nFirst 10 images:")
for image_path in val_images[:10]:
    print(image_path.name)

Number of validation images: 37

First 10 images:
makanan_106_jpg.rf.00945e253808ecda6ded3d90d9b0bead.jpg
makanan_135_jpg.rf.3881553ea2ccd286dd3d5b2679120e69.jpg
makanan_143_jpg.rf.2ff64ec79d0d26fa8cef1123bcece8b7.jpg
makanan_14_jpg.rf.543b4d9f5745c1e5de35a5623fdc9cfc.jpg
makanan_168_jpg.rf.7c584db5ee106da0d1ae22412b02c273.jpg
makanan_174_jpg.rf.cead4cbf6c36fa77e6d3169ff0631a63.jpg
makanan_179_jpg.rf.ee3258e0b7fc2d8edae392fa0d706187.jpg
makanan_184_jpg.rf.33695db8622ed5d832ef4081e7f390d4.jpg
makanan_185_jpg.rf.ea4ba015025b43046ab0af6547844915.jpg
makanan_187_jpg.rf.45ed283a5219d4d373d3cf304de69931.jpg


In [26]:
TEST_IMAGE = val_images[0]

In [27]:
results = model.predict(
    source=str(TEST_IMAGE),
    imgsz=640,
    device=DEVICE,
    conf=0.25,
    verbose=False
)

In [28]:
results[0].show()

# 5. BASELINE MODEL 

In [29]:
baseline_results = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    batch=8,
    device=DEVICE,
    split="val",
    plots=True,
    verbose=True,
    name="baseline_yolo11n_seg"
)

Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n-seg summary (fused): 113 layers, 2,868,664 parameters, 0 gradients, 9.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 199.564.9 MB/s, size: 57.8 KB)
val: Scanning D:\PREP_INTERN\nutrivision_pro\datasets\valid\labels... 37 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 37/37 1.1Kit/s 0.0s
val: New cache created: D:\PREP_INTERN\nutrivision_pro\datasets\valid\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.3s0.6s
                   all         37        534          0          0          0          0          0          0          0          0
                person          3          9          0          0          0          0          0          0          0          0
               bicycle         17        116          0     

In [30]:
print(baseline_results)

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001FD09F127B0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,

In [31]:
print("Box mAP50:", baseline_results.box.map50)
print("Box mAP50-95:", baseline_results.box.map)

print("Mask mAP50:", baseline_results.seg.map50)
print("Mask mAP50-95:", baseline_results.seg.map)

Box mAP50: 0.0
Box mAP50-95: 0.0
Mask mAP50: 0.0
Mask mAP50-95: 0.0


In [32]:
print("Box Precision:", baseline_results.box.mp)
print("Box Recall:", baseline_results.box.mr)

print("Mask Precision:", baseline_results.seg.mp)
print("Mask Recall:", baseline_results.seg.mr)

Box Precision: 0.0
Box Recall: 0.0
Mask Precision: 0.0
Mask Recall: 0.0


In [33]:
baseline_summary = pd.DataFrame({
    "metric": [
        "Box Precision",
        "Box Recall",
        "Box mAP50",
        "Box mAP50-95",
        "Mask Precision",
        "Mask Recall",
        "Mask mAP50",
        "Mask mAP50-95"
    ],
    "value": [
        baseline_results.box.mp,
        baseline_results.box.mr,
        baseline_results.box.map50,
        baseline_results.box.map,
        baseline_results.seg.mp,
        baseline_results.seg.mr,
        baseline_results.seg.map50,
        baseline_results.seg.map
    ]
})

baseline_summary

,metric,value
0,Box Precision,0.0
1,Box Recall,0.0
2,Box mAP50,0.0
3,Box mAP50-95,0.0
4,Mask Precision,0.0
5,Mask Recall,0.0
6,Mask mAP50,0.0
7,Mask mAP50-95,0.0


In [34]:
RESULT_DIR = PROJECT_ROOT / "results" / "baseline_yolo11n_seg"

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Result directory:")
print(RESULT_DIR.resolve())

Result directory:
D:\PREP_INTERN\nutrivision_pro\results\baseline_yolo11n_seg


In [35]:
baseline_summary.to_csv(
    RESULT_DIR / "baseline_metrics.csv",
    index=False
)

In [36]:
experiment_config = {
    "model": MODEL_NAME,
    "task": "instance_segmentation",
    "dataset": "NutriVision",
    "num_classes": len(names),
    "split": "val",
    "imgsz": 640,
    "batch": 8,
    "device": str(DEVICE),
    "fine_tuning": False,
    "training": False
}

with open(
    RESULT_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(experiment_config, f, indent=4)

print("Experiment configuration saved.")

Experiment configuration saved.
